# Micron (MU) — Section 3: Stockholder Performance, re-anchored to 15 Sep 2026

**Purpose.** Rebuild the five-year shareholder-return analysis on the report's market-data date
(15 Sep 2026 close) instead of the March 2026 window in the earlier draft. Everything the section
quotes is computed here; nothing is typed in by hand.

**Window.** 30 Sep 2021 close → 15 Sep 2026 close (4.96 years). Sub-windows: 3-year from
29 Sep 2023, 1-year from 15 Sep 2025. All windows end on the same date as the valuation in §5.

**Data and basis (state this in the report).**

| Series | Source | Basis |
|---|---|---|
| MU | Interactive Brokers historical data (daily close) + cash-dividend records (ex-dates) | **Total return**, dividends reinvested at the ex-date close |
| DJIA, S&P 500, Nasdaq Composite, PHLX SOX | ETF proxies **DIA, SPY, ONEQ, SOXX** (same source) | **Total return** via ETF distributions; net of expense ratios of 0.09–0.21% p.a. |
| Peers: SK hynix, Samsung, Nanya, Kioxia, SanDisk, Western Digital | LSEG Workspace price history, USD-converted, pulled 15 Sep 2026 (`data + files/lseg pulls/peer price history/`) | **Price return in USD**, then an **approximate** dividend reinvestment using LSEG's fiscal-year-end gross dividend yields |
| Risk-free rate | FRED TB3MS (3-month T-bill, monthly) | Sharpe denominator / excess return |

The peer dividend adjustment is an approximation and is labelled as such in the exhibit. The exact
replacement is one LSEG TRTR pull per peer; the code is written so that dropping a TRTR series into
`peer_tr_override` replaces the approximation without touching anything else.

Data-quality notes: the SOXX price series is split-adjusted (3-for-1 on 7 Mar 2024) but the pre-split
distributions are not, so pre-split dividends are divided by 3 before reinvestment. The LSEG WDC series
is adjusted for the February 2025 SanDisk separation (its 31 Dec 2019 close is 47.96 against an
unadjusted 63.47), so WDC's return already includes the value distributed in SanDisk shares.


In [1]:
import json, glob, os
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import openpyxl

DATA = os.path.join('..', 'data + files')
TSR  = os.path.join(DATA, 'tsr')
PEER = os.path.join(DATA, 'lseg pulls', 'peer price history')
OUT  = os.path.join('..', 'Analysis Topics')
os.makedirs(OUT, exist_ok=True)

START_5Y = pd.Timestamp('2021-09-30')
START_3Y = pd.Timestamp('2023-09-29')
START_1Y = pd.Timestamp('2025-09-15')
END      = pd.Timestamp('2026-09-15')
pd.set_option('display.width', 160)

## 1. Load the daily series and build total-return indices

In [2]:
def load_ibkr(ticker):
    d = json.load(open(os.path.join(TSR, f'{ticker}_daily.json')))
    px = pd.Series(d['close'], index=pd.to_datetime([t[:10] for t in d['time']]), name=ticker).sort_index()
    px = px[~px.index.duplicated(keep='last')]
    divs = {}
    split_date, split_ratio = None, 1.0
    for a in d.get('corp_actions', []):
        if a['type'] == 'Splits':
            split_date, split_ratio = pd.Timestamp(a['date']), float(a['value'])
    for a in d.get('corp_actions', []):
        if a['type'] == 'CashDividends':
            dt, v = pd.Timestamp(a['date']), float(a['value'])
            if split_date is not None and dt < split_date:
                v = v / split_ratio          # prices are split-adjusted; pre-split cash amounts are not
            divs[dt] = divs.get(dt, 0.0) + v
    divs = pd.Series(divs).sort_index()
    return px, divs

def total_return_index(px, divs):
    # reinvest each dividend at the ex-date close: TR_t = TR_{t-1} * (P_t + D_t) / P_{t-1}
    d = divs.reindex(px.index).fillna(0.0)
    # dividends whose ex-date is not a trading day in the file roll to the next trading day
    missing = divs.index.difference(px.index)
    for dt in missing:
        nxt = px.index[px.index.searchsorted(dt)]
        d[nxt] += divs[dt]
    factor = (px + d) / px.shift(1)
    factor.iloc[0] = 1.0
    return factor.cumprod()

series = {}
for t in ['MU', 'SPY', 'DIA', 'SOXX', 'ONEQ']:
    px, divs = load_ibkr(t)
    series[t] = {'px': px, 'divs': divs, 'tr': total_return_index(px, divs)}
    print(f"{t:5s} {px.index[0].date()} → {px.index[-1].date()}  first {px.iloc[0]:.2f}  last {px.iloc[-1]:.2f}  dividends: {len(divs)}  total cash/share {divs.sum():.3f}")

MU    2021-09-20 → 2026-09-15  first 72.37  last 927.60  dividends: 20  total cash/share 2.325
SPY   2021-09-20 → 2026-09-15  first 434.04  last 757.39  dividends: 19  total cash/share 32.638
DIA   2021-09-20 → 2026-09-15  first 339.40  last 521.23  dividends: 59  total cash/share 32.586
SOXX  2021-09-20 → 2026-09-15  first 152.46  last 498.85  dividends: 22  total cash/share 7.671
ONEQ  2021-09-20 → 2026-09-15  first 57.23  last 102.47  dividends: 20  total cash/share 2.171


## 2. Peer series from the LSEG USD price-history exports

In [3]:
def load_lseg_price(fname):
    ws = openpyxl.load_workbook(os.path.join(PEER, fname), read_only=True, data_only=True).active
    rows = list(ws.iter_rows(values_only=True))
    hdr = next(i for i, r in enumerate(rows) if r and r[0] == 'Exchange Date')
    data = [(r[0], r[1]) for r in rows[hdr+1:] if r and isinstance(r[0], pd.Timestamp.__mro__[1]) or (r and hasattr(r[0], 'year'))]
    s = pd.Series({pd.Timestamp(d): float(p) for d, p in data if p is not None}).sort_index()
    return s

peer_files = {'SK hynix': 'sk-hynix.xlsx', 'Samsung Electronics': 'samsung.xlsx', 'Nanya Technology': 'nanya.xlsx',
              'Kioxia Holdings': 'kioxia.xlsx', 'SanDisk': 'sandisk.xlsx', 'Western Digital': 'western.xlsx'}
peers = {k: load_lseg_price(v) for k, v in peer_files.items()}
for k, s in peers.items():
    print(f"{k:22s} {s.index[0].date()} → {s.index[-1].date()}  n={len(s)}  last {s.iloc[-1]:.2f} USD")

# approximate dividend reinvestment: apply LSEG's fiscal-year-end gross yield on the FYE date
yields = pd.read_csv(os.path.join(TSR, 'peer_dividend_yields.csv'), parse_dates=['fiscal_year_end'])
peer_tr_override = {}   # name -> pd.Series of an LSEG TRTR index, if/when pulled; replaces the approximation

def peer_tr(name):
    if name in peer_tr_override:
        return peer_tr_override[name]
    px = peers[name]
    tr = px.copy()
    y = yields[yields.company == name].set_index('fiscal_year_end')['gross_dividend_yield_pct'] / 100.0
    factor = pd.Series(1.0, index=px.index)
    for dt, yv in y.items():
        if yv > 0 and dt >= px.index[0]:
            pos = px.index.searchsorted(dt)
            if pos < len(px.index):
                factor.iloc[pos] *= (1 + yv)
    return px * factor.cumprod() / factor.cumprod().iloc[0]

peer_tr_series = {k: peer_tr(k) for k in peers}

SK hynix               2020-01-02 → 2026-09-15  n=1646  last 1254.61 USD
Samsung Electronics    2020-01-02 → 2026-09-15  n=1646  last 184.48 USD
Nanya Technology       2019-12-31 → 2026-09-15  n=1630  last 14.66 USD
Kioxia Holdings        2024-12-18 → 2026-09-15  n=424  last 335.28 USD
SanDisk                2025-02-13 → 2026-09-15  n=398  last 1530.89 USD
Western Digital        2019-12-31 → 2026-09-15  n=1685  last 411.96 USD


## 3. Return table — 1-, 3- and 5-year, price and total return

In [4]:
def window_return(s, start, end):
    s = s.dropna()
    if s.index[0] > start + pd.Timedelta(days=7):
        return np.nan
    a = s[:start].iloc[-1]; b = s[:end].iloc[-1]
    return b / a - 1

def annualise(r, start, end):
    if pd.isna(r): return np.nan
    yrs = (end - start).days / 365.25
    return (1 + r) ** (1 / yrs) - 1

def yrs(start, end): return (end - start).days / 365.25

rows = []
def add_row(name, group, px_s, tr_s):
    r = {'Security': name, 'Group': group}
    for lab, st in [('1Y', START_1Y), ('3Y', START_3Y), ('5Y', START_5Y)]:
        pr = window_return(px_s, st, END); tr = window_return(tr_s, st, END)
        r[f'{lab} price'] = pr; r[f'{lab} TR'] = tr
        r[f'{lab} TR ann.'] = tr if lab == '1Y' else annualise(tr, st, END)
    rows.append(r)

add_row('Micron (MU)', 'Subject', series['MU']['px'], series['MU']['tr'])
for name, tk in [('DJIA (via DIA)', 'DIA'), ('S&P 500 (via SPY)', 'SPY'), ('Nasdaq Composite (via ONEQ)', 'ONEQ'), ('PHLX Semiconductor (via SOXX)', 'SOXX')]:
    add_row(name, 'Benchmark', series[tk]['px'], series[tk]['tr'])
for k in peers:
    add_row(k, 'Peer', peers[k], peer_tr_series[k])

tbl = pd.DataFrame(rows).set_index('Security')
pct = lambda x: '' if pd.isna(x) else f'{x*100:+.1f}%'
show = tbl[['Group', '1Y TR', '3Y TR ann.', '5Y TR ann.', '5Y TR', '5Y price']].copy()
for c in show.columns[1:]: show[c] = show[c].map(pct)
print(f"Windows: 5Y {START_5Y.date()}→{END.date()} ({yrs(START_5Y, END):.2f} yrs); 3Y {START_3Y.date()}→; 1Y {START_1Y.date()}→")
show

Windows: 5Y 2021-09-30→2026-09-15 (4.96 yrs); 3Y 2023-09-29→; 1Y 2025-09-15→


,Group,1Y TR,3Y TR ann.,5Y TR ann.,5Y TR,5Y price
Security,,,,,,
Micron (MU),Subject,+488.9%,+142.5%,+68.7%,+1238.5%,+1206.8%
DJIA (via DIA),Benchmark,+15.0%,+17.9%,+11.0%,+67.6%,+54.1%
S&P 500 (via SPY),Benchmark,+15.9%,+22.7%,+13.6%,+88.3%,+76.5%
Nasdaq Composite (via ONEQ),Benchmark,+17.1%,+26.5%,+13.7%,+88.8%,+82.4%
PHLX Semiconductor (via SOXX),Benchmark,+94.5%,+48.3%,+28.6%,+248.6%,+235.6%
SK hynix,Peer,+426.9%,+150.0%,+73.4%,+1431.5%,+1342.5%
Samsung Electronics,Peer,+238.1%,+57.9%,+27.4%,+232.6%,+194.8%
Nanya Technology,Peer,+601.6%,+96.3%,+47.8%,+592.9%,+518.8%
Kioxia Holdings,Peer,+1015.0%,,,,


In [5]:
# Peer median (memory peers with a full 5-year history) and MU rank
full = [k for k in peers if not pd.isna(tbl.loc[k, '5Y TR ann.'])]
print('Peers with full 5-year history:', full)
for lab in ['1Y TR', '3Y TR ann.', '5Y TR ann.']:
    pv = tbl.loc[[k for k in peers if not pd.isna(tbl.loc[k, lab])], lab]
    mu = tbl.loc['Micron (MU)', lab]
    rank = 1 + (pv > mu).sum()
    print(f"{lab:11s} MU {mu*100:+.1f}%   peer median {pv.median()*100:+.1f}%   MU rank {rank} of {len(pv)+1}")

Peers with full 5-year history: ['SK hynix', 'Samsung Electronics', 'Nanya Technology', 'Western Digital']
1Y TR       MU +488.9%   peer median +514.3%   MU rank 4 of 7
3Y TR ann.  MU +142.5%   peer median +113.8%   MU rank 2 of 5
5Y TR ann.  MU +68.7%   peer median +52.9%   MU rank 2 of 5


## 4. Indexed total-return chart (Exhibit for the report)

In [6]:
base = START_5Y
def indexed(tr):
    tr = tr[tr.index >= base]
    return 100 * tr / tr.iloc[0]

palette = {'MU': '#2C5AA0', 'SOXX': '#D4772A', 'ONEQ': '#1F9E89', 'SPY': '#8E5BC7', 'DIA': '#B0405A'}
labels  = {'MU': 'Micron (TR)', 'SOXX': 'PHLX SOX (SOXX, TR)', 'ONEQ': 'Nasdaq Comp. (ONEQ, TR)', 'SPY': 'S&P 500 (SPY, TR)', 'DIA': 'DJIA (DIA, TR)'}

fig, ax = plt.subplots(figsize=(7.0, 3.4), dpi=200)
for tk in ['DIA', 'SPY', 'ONEQ', 'SOXX', 'MU']:
    s = indexed(series[tk]['tr'])
    ax.plot(s.index, s.values, color=palette[tk], lw=2.0 if tk == 'MU' else 1.4, label=labels[tk])
    dy = {'SPY': -5, 'ONEQ': 5}.get(tk, 0)
    ax.annotate(f"{s.iloc[-1]:,.0f}", (s.index[-1], s.iloc[-1]), xytext=(4, dy), textcoords='offset points',
                va='center', fontsize=7, color='#333333')
ax.set_yscale('log')
ax.set_yticks([50, 100, 200, 400, 800, 1600])
ax.get_yaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.set_ylim(45, 2000)
ax.axhline(100, color='#999999', lw=0.6, ls=':')
ax.set_ylabel('Indexed total return (30 Sep 2021 = 100, log scale)', fontsize=7.5)
ax.tick_params(labelsize=7.5)
ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
ax.grid(axis='y', color='#e5e5e5', lw=0.5)
ax.legend(fontsize=7, frameon=False, loc='upper left', ncol=1)
# annotate the trough and the peak of the MU series
mu_i = indexed(series['MU']['tr'])
tmin = mu_i.idxmin(); tmax = mu_i.idxmax()
ax.annotate(f"trough {mu_i.min():.0f}\n{tmin:%b %Y}", (tmin, mu_i.min()), xytext=(0, -26), textcoords='offset points', ha='center', fontsize=6.5, color='#333333', arrowprops=dict(arrowstyle='-', color='#999999', lw=0.6))
ax.annotate(f"peak {mu_i.max():,.0f}\n{tmax:%d %b %Y}", (tmax, mu_i.max()), xytext=(-30, 6), textcoords='offset points', ha='center', fontsize=6.5, color='#333333')
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'exhibit_section3_tsr.png'), dpi=200)
print('saved', os.path.join(OUT, 'exhibit_section3_tsr.png'))
print(f"MU indexed: trough {mu_i.min():.1f} on {tmin.date()}, peak {mu_i.max():.1f} on {tmax.date()}, end {mu_i.iloc[-1]:.1f}")

saved ..\Analysis Topics\exhibit_section3_tsr.png
MU indexed: trough 69.2 on 2022-09-26, peak 1750.9 on 2026-06-25, end 1338.5


## 5. Risk-adjusted measures over the five-year window (monthly total returns)

In [7]:
tb = pd.read_csv(os.path.join(TSR, 'TB3MS.csv'), parse_dates=['observation_date']).set_index('observation_date')['TB3MS'] / 100
rf_avg = tb[(tb.index >= '2021-10-01') & (tb.index <= '2026-08-01')].mean()
print(f"Average 3-month T-bill Oct-2021..Aug-2026 (FRED TB3MS): {rf_avg*100:.2f}%")

def monthly_tr(tr):
    tr = tr[(tr.index >= START_5Y) & (tr.index <= END)]
    m = tr.resample('ME').last()
    m.iloc[-1] = tr.iloc[-1]
    return m.pct_change().dropna()

def risk_stats(tr, mkt_tr=None):
    m = monthly_tr(tr)
    ann_vol = m.std(ddof=1) * np.sqrt(12)
    cum = window_return(tr, START_5Y, END); ann = annualise(cum, START_5Y, END)
    sharpe = (ann - rf_avg) / ann_vol
    d = tr[(tr.index >= START_5Y) & (tr.index <= END)]
    dd = d / d.cummax() - 1
    mdd = dd.min(); mdd_date = dd.idxmin(); peak_date = d[:mdd_date].idxmax()
    out = {'Ann. TR': ann, 'Ann. vol': ann_vol, 'Sharpe': sharpe, 'Max drawdown': mdd, 'DD peak': peak_date.date(), 'DD trough': mdd_date.date()}
    if mkt_tr is not None:
        mm = monthly_tr(mkt_tr).reindex(m.index)
        beta = np.cov(m, mm, ddof=1)[0, 1] / mm.var(ddof=1)
        corr = np.corrcoef(m, mm)[0, 1]
        out['Beta vs S&P 500 (60m)'] = beta; out['Corr'] = corr
    return out

risk = pd.DataFrame({
    'Micron (MU)': risk_stats(series['MU']['tr'], series['SPY']['tr']),
    'DJIA (DIA)': risk_stats(series['DIA']['tr'], series['SPY']['tr']),
    'S&P 500 (SPY)': risk_stats(series['SPY']['tr'], series['SPY']['tr']),
    'Nasdaq Comp. (ONEQ)': risk_stats(series['ONEQ']['tr'], series['SPY']['tr']),
    'PHLX SOX (SOXX)': risk_stats(series['SOXX']['tr'], series['SPY']['tr']),
    'SK hynix': risk_stats(peer_tr_series['SK hynix'], series['SPY']['tr']),
    'Samsung Electronics': risk_stats(peer_tr_series['Samsung Electronics'], series['SPY']['tr']),
    'Nanya Technology': risk_stats(peer_tr_series['Nanya Technology'], series['SPY']['tr']),
    'Western Digital': risk_stats(peer_tr_series['Western Digital'], series['SPY']['tr']),
}).T
fmt = risk.copy()
for c in ['Ann. TR', 'Ann. vol', 'Max drawdown']: fmt[c] = fmt[c].map(lambda x: f'{x*100:+.1f}%')
for c in ['Sharpe', 'Beta vs S&P 500 (60m)', 'Corr']: fmt[c] = fmt[c].map(lambda x: f'{x:.2f}')
fmt

Average 3-month T-bill Oct-2021..Aug-2026 (FRED TB3MS): 3.78%


,Ann. TR,Ann. vol,Sharpe,Max drawdown,DD peak,DD trough,Beta vs S&P 500 (60m),Corr
Micron (MU),+68.7%,+68.0%,0.96,-57.6%,2024-06-18,2025-04-04,2.24,0.52
DJIA (DIA),+11.0%,+14.9%,0.48,-20.8%,2022-01-04,2022-09-30,0.86,0.91
S&P 500 (SPY),+13.6%,+15.7%,0.63,-24.5%,2022-01-03,2022-10-12,1.00,1.00
Nasdaq Comp. (ONEQ),+13.7%,+20.3%,0.49,-35.2%,2021-11-19,2022-12-28,1.22,0.94
PHLX SOX (SOXX),+28.6%,+37.6%,0.66,-45.8%,2021-12-27,2022-10-14,1.81,0.75
SK hynix,+73.4%,+70.1%,0.99,-51.1%,2026-06-22,2026-07-30,2.28,0.51
Samsung Electronics,+27.4%,+47.1%,0.50,-44.7%,2022-01-12,2022-09-29,1.47,0.49
Nanya Technology,+47.8%,+78.1%,0.56,-71.8%,2022-02-18,2025-01-13,1.67,0.34
Western Digital,+58.1%,+59.5%,0.91,-55.3%,2022-01-04,2022-12-28,2.13,0.56


## 6. Phase decomposition of the MU total return and the dollar-value summary

In [8]:
mu_tr = series['MU']['tr']; mu_px = series['MU']['px']
def lvl(s, dt): return s[:pd.Timestamp(dt)].iloc[-1]
phases = [('Sep 2021 → Dec 2022 (memory downturn)', '2021-09-30', '2022-12-30'),
          ('Dec 2022 → Dec 2024 (recovery and false dawn)', '2022-12-30', '2024-12-31'),
          ('Dec 2024 → 25 Jun 2026 (HBM re-rating to peak)', '2024-12-31', '2026-06-25'),
          ('25 Jun 2026 → 15 Sep 2026 (correction)', '2026-06-25', '2026-09-15')]
for name, a, b in phases:
    r = lvl(mu_tr, b) / lvl(mu_tr, a) - 1
    print(f"{name:52s} price {lvl(mu_px,a):8.2f} → {lvl(mu_px,b):8.2f}   TR {r*100:+7.1f}%")
print()
print('Key closes: ', {d: round(lvl(mu_px, d), 2) for d in ['2021-09-30', '2022-12-30', '2023-09-29', '2024-12-31', '2025-09-15', '2026-06-25', '2026-09-14', '2026-09-15']})
print('MU intraday high in window:', json.load(open(os.path.join(TSR, 'MU_daily.json')))['high'].__class__.__name__)
d = json.load(open(os.path.join(TSR, 'MU_daily.json')))
hi = pd.Series(d['high'], index=pd.to_datetime([t[:10] for t in d['time']]))
print(f"Intraday high {hi.max():.2f} on {hi.idxmax().date()}")
for nm, tk in [('MU', 'MU'), ('DIA', 'DIA'), ('SPY', 'SPY'), ('ONEQ', 'ONEQ'), ('SOXX', 'SOXX')]:
    r = window_return(series[tk]['tr'], START_5Y, END)
    print(f"$10,000 in {nm:4s} on {START_5Y.date()} → ${10000*(1+r):,.0f} on {END.date()}  (TR {r*100:+.1f}%)")
mu5 = window_return(series['MU']['tr'], START_5Y, END); mu5p = window_return(series['MU']['px'], START_5Y, END)
print(f"MU dividend contribution to 5Y TR: {(mu5-mu5p)*100:.1f} pp of {mu5*100:.1f} pp")

Sep 2021 → Dec 2022 (memory downturn)                price    70.98 →    49.98   TR   -29.0%
Dec 2022 → Dec 2024 (recovery and false dawn)        price    49.98 →    84.16   TR   +70.3%
Dec 2024 → 25 Jun 2026 (HBM re-rating to peak)       price    84.16 →  1213.56   TR +1347.4%
25 Jun 2026 → 15 Sep 2026 (correction)               price  1213.56 →   927.60   TR   -23.6%

Key closes:  {'2021-09-30': np.float64(70.98), '2022-12-30': np.float64(49.98), '2023-09-29': np.float64(68.03), '2024-12-31': np.float64(84.16), '2025-09-15': np.float64(157.77), '2026-06-25': np.float64(1213.56), '2026-09-14': np.float64(924.03), '2026-09-15': np.float64(927.6)}
MU intraday high in window: list
Intraday high 1255.00 on 2026-06-25
$10,000 in MU   on 2021-09-30 → $133,851 on 2026-09-15  (TR +1238.5%)
$10,000 in DIA  on 2021-09-30 → $16,764 on 2026-09-15  (TR +67.6%)
$10,000 in SPY  on 2021-09-30 → $18,826 on 2026-09-15  (TR +88.3%)
$10,000 in ONEQ on 2021-09-30 → $18,878 on 2026-09-15  (TR +88.8%)
$10,0

## 7. Export the numbers the report quotes

In [9]:
tbl.to_csv(os.path.join(TSR, 'section3_returns.csv'))
risk.to_csv(os.path.join(TSR, 'section3_risk.csv'))
print('written section3_returns.csv and section3_risk.csv')

written section3_returns.csv and section3_risk.csv
